# Kinematics Calculator
This notebook is used for deriving and calculating the rig's kinematics

## Deriving the symbolic transformations Base -> End of Beam

In [2]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib
import sympy as sp
import plotly.io as pio

# Declaring the symbols
d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg = sp.symbols(
    r'd1 theta_y theta_p a_b h_b c_{boff} c_{h} c_{boc} \beta_{cam}',
    real=True
)

# Defining the DH parameters (with additional offsets on the links)
# Please note two more translation parameters are added to the DH table to account for the offsets in the links 
# The columns means as follows: [link number, link length, link twist, link offset (3 axes) joint angle]
DH = sp.Matrix([[1, 0, 0, 0, 0, d1, theta_y], \
            [2, 0, sp.pi/2, 0, 0, 0, theta_p], \
            [3, a_b, 0, 0, h_b, 0, 0], \
            [4, 0, 0, camera_beam_offset, camera_height, camera_offset_from_beam_centre, 0],
            [5, 0, sp.pi*camera_angle_deg/180, 0, 0, 0, 0]])

DH


Matrix([
[1,   0,                  0,        0,     0,      d1, theta_y],
[2,   0,               pi/2,        0,     0,       0, theta_p],
[3, a_b,                  0,        0,   h_b,       0,       0],
[4,   0,                  0, c_{boff}, c_{h}, c_{boc},       0],
[5,   0, pi*\beta_{cam}/180,        0,     0,       0,       0]])

In [3]:
# Calculating the individual transformation matrices using the DH parameters
N = len(DH[:, 0])

def create_transormation_string(superscript, subscript):
    return sp.symbols("{}^{" + str(superscript) + "}_" + str(subscript) + "T")


Full_forward_kinematics = sp.Matrix([[1, 0, 0, 0],\
                                  [0, 1, 0, 0],\
                                  [0, 0, 1, 0],
                                  [0, 0, 0, 1]])

Individual_transformation_matrices = []

for i in range(N):
    # Extracting parameters from the current link
    m = DH[i, 0]
    a_mm1 = DH[i, 1] #mm1 means m minus 1
    alpha_mm1 = DH[i, 2]
    tm = DH[i, 6]
    translate_i = DH[i, 3]
    translate_j = DH[i, 4]
    translate_k = DH[i, 5]



    print(f"=== LINK {m} ===")

    T_mm1_R = sp.Matrix([[1, 0, 0, 0], \
                         [0, sp.cos(alpha_mm1), -sp.sin(alpha_mm1), 0], \
                         [0, sp.sin(alpha_mm1), sp.cos(alpha_mm1), 0], \
                         [0, 0, 0, 1]])
    display(create_transormation_string(m-1, 'R'), T_mm1_R)

    T_R_P = sp.Matrix([[1, 0, 0, a_mm1], \
                       [0, 1, 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('R', 'P'), T_R_P)

    T_P_Q = sp.Matrix([[sp.cos(tm), -sp.sin(tm), 0, 0], \
                       [sp.sin(tm), sp.cos(tm), 0, 0], \
                       [0, 0, 1, 0], \
                       [0, 0, 0, 1]])   
    display(create_transormation_string('P', 'Q'), T_P_Q)

    T_Q_m = sp.Matrix([[1, 0, 0, translate_i], \
                       [0, 1, 0, translate_j], \
                       [0, 0, 1, translate_k], \
                       [0, 0, 0, 1]])
    display(create_transormation_string('Q', m), T_Q_m)

    # Calculating the transformation matrix from frame m-1 to frame m
    T_mm1_m = sp.simplify(T_mm1_R * T_R_P * T_P_Q * T_Q_m)
    Individual_transformation_matrices.append(T_mm1_m)

    Full_forward_kinematics = sp.simplify(Full_forward_kinematics * T_mm1_m)

    display(create_transormation_string(m-1, m), T_mm1_m)

print("=== Full Forward Kinematics ===")
display(sp.simplify(Full_forward_kinematics))

print("=== Individual Transformation Matrices ===")
display(Individual_transformation_matrices)



=== LINK 1 ===


{}^{0}_RT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

{}^{R}_PT

Matrix([
[1, 0, 0, 0],
[0, 1, 0, 0],
[0, 0, 1, 0],
[0, 0, 0, 1]])

...

Matrix([
[1,                       0,                        0, 0],
[0, cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), 0],
[0, sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180), 0],
[0,                       0,                        0, 1]])

=== Full Forward Kinematics ===


Matrix([
[cos(theta_p)*cos(theta_y), -sin(theta_p)*cos(theta_y)*cos(pi*\beta_{cam}/180) + sin(theta_y)*sin(pi*\beta_{cam}/180), sin(theta_p)*sin(pi*\beta_{cam}/180)*cos(theta_y) + sin(theta_y)*cos(pi*\beta_{cam}/180),  c_{boc}*sin(theta_y) + c_{boff}*cos(theta_p)*cos(theta_y) - c_{h}*sin(theta_p)*cos(theta_y) + (a_b*cos(theta_p) - h_b*sin(theta_p))*cos(theta_y)],
[sin(theta_y)*cos(theta_p), -sin(theta_p)*sin(theta_y)*cos(pi*\beta_{cam}/180) - sin(pi*\beta_{cam}/180)*cos(theta_y), sin(theta_p)*sin(theta_y)*sin(pi*\beta_{cam}/180) - cos(theta_y)*cos(pi*\beta_{cam}/180), -c_{boc}*cos(theta_y) + c_{boff}*sin(theta_y)*cos(theta_p) - c_{h}*sin(theta_p)*sin(theta_y) + (a_b*cos(theta_p) - h_b*sin(theta_p))*sin(theta_y)],
[             sin(theta_p),                                                      cos(theta_p)*cos(pi*\beta_{cam}/180),                                                    -sin(pi*\beta_{cam}/180)*cos(theta_p),                                                             a_b*sin(

=== Individual Transformation Matrices ===


[Matrix([
 [cos(theta_y), -sin(theta_y), 0,  0],
 [sin(theta_y),  cos(theta_y), 0,  0],
 [           0,             0, 1, d1],
 [           0,             0, 0,  1]]),
 Matrix([
 [cos(theta_p), -sin(theta_p),  0, 0],
 [           0,             0, -1, 0],
 [sin(theta_p),  cos(theta_p),  0, 0],
 [           0,             0,  0, 1]]),
 Matrix([
 [1, 0, 0, a_b],
 [0, 1, 0, h_b],
 [0, 0, 1,   0],
 [0, 0, 0,   1]]),
 Matrix([
 [1, 0, 0, c_{boff}],
 [0, 1, 0,    c_{h}],
 [0, 0, 1,  c_{boc}],
 [0, 0, 0,        1]]),
 Matrix([
 [1,                       0,                        0, 0],
 [0, cos(pi*\beta_{cam}/180), -sin(pi*\beta_{cam}/180), 0],
 [0, sin(pi*\beta_{cam}/180),  cos(pi*\beta_{cam}/180), 0],
 [0,                       0,                        0, 1]])]

In [7]:
# Creating a function

parameters = (d1, theta_y, theta_p, a_b, h_b, camera_beam_offset, camera_height, camera_offset_from_beam_centre, camera_angle_deg)

_var_d1 = 0.353
_var_theta_y = 0.0
_var_theta_p = 0.0
_var_a_b = 0.83
_var_h_b = 0.0761
_var_camera_beam_offset = -0.043
_var_camera_height = 0.029
_var_camera_offset_from_beam_centre = 0.04
_var_camera_angle_deg = 40

_ffk = sp.lambdify(parameters, Full_forward_kinematics, modules='numpy')

_fT01 = sp.lambdify(parameters, Individual_transformation_matrices[0], modules='numpy')
_fT12 = sp.lambdify(parameters, Individual_transformation_matrices[1], modules='numpy')
_fT23 = sp.lambdify(parameters, Individual_transformation_matrices[2], modules='numpy')
_fT34 = sp.lambdify(parameters, Individual_transformation_matrices[3], modules='numpy')
_ftT45 = sp.lambdify(parameters, Individual_transformation_matrices[4], modules='numpy')

# Example calculation
T_cam = _ffk(
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

print("=== Full Forward Kinematics ===")
print(T_cam)

# Checking
print("=== Checking the individual transformation matrices ===")

_fkin_args = (
    _var_d1,
    _var_theta_y,
    _var_theta_p,
    _var_a_b,
    _var_h_b,
    _var_camera_beam_offset,
    _var_camera_height,
    _var_camera_offset_from_beam_centre,
    _var_camera_angle_deg
)

T01 = _fT01(*_fkin_args)
T12 = _fT12(*_fkin_args)
T23 = _fT23(*_fkin_args)
T34 = _fT34(*_fkin_args)
T45 = _ftT45(*_fkin_args)

#Matrix multiplication
T_cam_check = T01 @ T12 @ T23 @ T34 @ T45

print(T_cam_check)
matrices_correct = False

if np.allclose(T_cam, T_cam_check):
    print("The individual transformation matrices are correct.")
    matrices_correct = True
else:
    raise ValueError("The individual transformation matrices are not correct. Please check the calculations.")


if matrices_correct:
    Base_coords = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    Frame_coords = [Base_coords]

    # Calculating the coordinates of each frame
    F1_coords = T01 @ Base_coords
    F2_coords = T01 @ T12 @ Base_coords
    F3_coords = T01 @ T12 @ T23 @ Base_coords
    F4_coords = T01 @ T12 @ T23 @ T34 @ Base_coords
    Fcam_coords = T01 @ T12 @ T23 @ T34 @ T45 @ Base_coords

    Frame_coords.append(F1_coords)
    Frame_coords.append(F2_coords)
    Frame_coords.append(F3_coords)
    Frame_coords.append(F4_coords)
    Frame_coords.append(Fcam_coords)

# Plot frame origins and coordinate systems in 3D using Plotly
origins = np.array([frame[:3, 3] for frame in Frame_coords], dtype=float)
rotations = np.array([frame[:3, :3] for frame in Frame_coords], dtype=float)

fig = go.Figure()

# Plot frame origins
fig.add_trace(go.Scatter3d(
    x=origins[:, 0],
    y=origins[:, 1],
    z=origins[:, 2],
    mode='markers+lines',
    marker=dict(size=5),
    line=dict(color='black', width=2),
    name='Frame origins'
))

# Plot coordinate axes for each frame
axis_length = 0.1
axis_colors = ['red', 'green', 'blue']
axis_names = ['X axis', 'Y axis', 'Z axis']

for axis in range(3):
    x_values, y_values, z_values = [], [], []

    for origin, rotation in zip(origins, rotations):
        endpoint = origin + axis_length * rotation[:, axis]

        x_values.extend([origin[0], endpoint[0], None])
        y_values.extend([origin[1], endpoint[1], None])
        z_values.extend([origin[2], endpoint[2], None])

    fig.add_trace(go.Scatter3d(
        x=x_values,
        y=y_values,
        z=z_values,
        mode='lines',
        line=dict(color=axis_colors[axis], width=4),
        name=axis_names[axis]
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='3D Plot of Forward Kinematics'
)

fig.show()

=== Full Forward Kinematics ===
[[ 1.          0.          0.          0.787     ]
 [ 0.         -0.64278761 -0.76604444 -0.04      ]
 [ 0.          0.76604444 -0.64278761  0.4581    ]
 [ 0.          0.          0.          1.        ]]
=== Checking the individual transformation matrices ===
[[ 1.          0.          0.          0.787     ]
 [ 0.         -0.64278761 -0.76604444 -0.04      ]
 [ 0.          0.76604444 -0.64278761  0.4581    ]
 [ 0.          0.          0.          1.        ]]
The individual transformation matrices are correct.


In [ ]:
# Learning how to project points onto the camera

Example_point_in_base_frame = np.array([[0.5], [0.1], [0.2], [1]])  # Homogeneous coordinates

# Inverting the camera's transformation matrix to get the transformation from the camera to the base frame
T_cam_to_base = np.linalg.inv(T_cam)
print("=== Transformation from camera to base frame ===")
print(T_cam_to_base)

#TODO Have these import in code, these have been now looked up from the camera calibration steam
# Image resolution
resolution = np.array([640, 480])  # [width, height]

# Camera intrinsic matrix
K = np.array([
    [461.8016663,   0.0,         324.1998901, 0.0],
    [0.0,           461.7047424, 242.8716125, 0.0],
    [0.0,           0.0,           1.0, 0.0]
])

# Distortion coefficients: [k1, k2, t1, t2, k3]
dist_coeffs = np.array([
     0.0056061,
    -0.0481728,
    -0.00040035,
     0.00041010,
     0.0330434
])

# For this single point
Example_point_in_camera_frame = T_cam_to_base @ Example_point_in_base_frame
print("=== Example point in camera frame ===")
print(Example_point_in_camera_frame)
pixel_coords_homogeneous = K @ T_cam_to_base @ Example_point_in_base_frame
pixel_coords = np.rint(pixel_coords_homogeneous[:2] / pixel_coords_homogeneous[2]).astype(int)
print("=== Pixel coordinates of the example point ===")
print(pixel_coords)



In [1]:
# ============================================================
# VISUALISATION
# Uses the results already calculated above.
# Does NOT recalculate or modify the kinematics/projection.
# ============================================================

from matplotlib.patches import Rectangle


# ------------------------------------------------------------
# 1. 3D ROBOT / CAMERA FRAMES + EXAMPLE POINT
# ------------------------------------------------------------

frame_labels = [
    "Base",
    "F1",
    "F2",
    "F3",
    "F4",
    "Camera"
]

# Origins and rotations already expressed in the base frame
origins = np.array([
    np.asarray(frame[:3, 3], dtype=float).reshape(3)
    for frame in Frame_coords
])

rotations = [
    np.asarray(frame[:3, :3], dtype=float)
    for frame in Frame_coords
]

# Example point is already defined in homogeneous base coordinates
point_base = np.asarray(
    Example_point_in_base_frame[:3],
    dtype=float
).reshape(3)


fig3d = go.Figure()


# -------------------------
# Connect frame origins
# -------------------------

fig3d.add_trace(go.Scatter3d(
    x=origins[:, 0],
    y=origins[:, 1],
    z=origins[:, 2],

    mode="lines+markers+text",

    text=frame_labels,
    textposition="top center",

    marker=dict(size=5),
    line=dict(width=3),

    name="Robot / camera frames"
))


# -------------------------
# Draw coordinate axes
# -------------------------

axis_length = 0.1

axis_colors = [
    "red",      # X
    "green",    # Y
    "blue"      # Z
]

axis_names = [
    "X axes",
    "Y axes",
    "Z axes"
]


for axis in range(3):

    x_values = []
    y_values = []
    z_values = []

    for origin, rotation in zip(origins, rotations):

        endpoint = (
            origin
            + axis_length * rotation[:, axis]
        )

        x_values.extend([
            origin[0],
            endpoint[0],
            None
        ])

        y_values.extend([
            origin[1],
            endpoint[1],
            None
        ])

        z_values.extend([
            origin[2],
            endpoint[2],
            None
        ])

    fig3d.add_trace(go.Scatter3d(
        x=x_values,
        y=y_values,
        z=z_values,

        mode="lines",

        line=dict(
            color=axis_colors[axis],
            width=4
        ),

        name=axis_names[axis]
    ))


# -------------------------
# Plot example 3D point
# -------------------------

fig3d.add_trace(go.Scatter3d(
    x=[point_base[0]],
    y=[point_base[1]],
    z=[point_base[2]],

    mode="markers+text",

    text=["Example point"],
    textposition="top center",

    marker=dict(size=8),

    name="Example point"
))


# -------------------------
# Draw line from camera
# to the example point
# -------------------------

camera_origin = origins[-1]

fig3d.add_trace(go.Scatter3d(
    x=[
        camera_origin[0],
        point_base[0]
    ],

    y=[
        camera_origin[1],
        point_base[1]
    ],

    z=[
        camera_origin[2],
        point_base[2]
    ],

    mode="lines",

    line=dict(
        width=2,
        dash="dash"
    ),

    name="Camera → point"
))


# -------------------------
# Plot formatting
# -------------------------

fig3d.update_layout(

    title="Robot kinematic frames and example point",

    scene=dict(

        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]",

        aspectmode="data"
    )
)

fig3d.show()



# ------------------------------------------------------------
# 2. CAMERA RGB IMAGE PLANE + PROJECTED POINT
# ------------------------------------------------------------

# Use the already-calculated pixel coordinates
pixel_flat = np.asarray(
    pixel_coords,
    dtype=float
).reshape(-1)

u = pixel_flat[0]
v = pixel_flat[1]

width = int(resolution[0])
height = int(resolution[1])


# Check whether the calculated projection lies within the image
inside_image = (
    0 <= u < width
    and
    0 <= v < height
)


fig2d, ax = plt.subplots(figsize=(8, 6))


# Draw the camera image boundary
rgb_frame = Rectangle(
    (0, 0),
    width,
    height,
    fill=False,
    linewidth=2,
    label=f"RGB frame ({width} × {height})"
)

ax.add_patch(rgb_frame)


# Plot projected point
ax.scatter(
    u,
    v,
    s=100,
    marker="x",
    label=f"Projected point ({u:.0f}, {v:.0f})"
)


# ------------------------------------------------------------
# Automatically expand plot if the projected point lies
# outside the camera image.
#
# This DOES NOT modify/clamp the calculated pixel coordinate.
# ------------------------------------------------------------

padding_x = max(20.0, 0.05 * width)
padding_y = max(20.0, 0.05 * height)

x_min = min(
    -padding_x,
    u - padding_x
)

x_max = max(
    width + padding_x,
    u + padding_x
)

y_min = min(
    -padding_y,
    v - padding_y
)

y_max = max(
    height + padding_y,
    v + padding_y
)


ax.set_xlim(
    x_min,
    x_max
)

# Image coordinates have Y pointing downward
ax.set_ylim(
    y_max,
    y_min
)

ax.set_aspect(
    "equal",
    adjustable="box"
)

ax.set_xlabel("u [pixels]")
ax.set_ylabel("v [pixels]")

ax.grid(True)
ax.legend()


if inside_image:

    ax.set_title(
        "RGB image plane — projected point is inside image"
    )

else:

    ax.set_title(
        "RGB image plane — projected point is OUTSIDE image"
    )


plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Useful numerical summary
# ------------------------------------------------------------

print("=== Visualisation summary ===")

print(
    f"Example point in base frame: "
    f"{point_base}"
)

print(
    f"Projected pixel coordinates: "
    f"u = {u:.2f}, v = {v:.2f}"
)

print(
    f"Inside {width} x {height} RGB frame: "
    f"{inside_image}"
)

NameError: name 'np' is not defined